# 08 Colab Extract — NEX-GDDP-CMIP6 Monthly Climatologies

Run this notebook in **Google Colab** (fast internet, modern Python).
Saves one small CSV per model/scenario/period to Google Drive.
Download those CSVs to `../data/climate_projections/` on your laptop,
then continue in `08_climate_projections.ipynb`.

**Upload to Colab:** File → Upload notebook → select this file  
**Runtime:** CPU is fine, no GPU needed.  
**Expected time:** ~30–60 min total for all 25 CSVs.

In [1]:
# ── Colab setup: mount Drive + install packages ───────────────────────────────
import sys
~sys.executable{pip -m install google.colab}
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'xarray', 's3fs', 'h5netcdf', 'netcdf4'])

import os
OUT_DIR = '/content/drive/MyDrive/ee_cmip6_exports'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output directory: {OUT_DIR}')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ── Imports and config ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
import s3fs
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

MODELS = [
    'ACCESS-CM2',
    'GFDL-ESM4',
    'MPI-ESM1-2-HR',
    'NorESM2-MM',
    'IPSL-CM6A-LR',
]

# Grid label varies by model in NEX-GDDP-CMIP6
GRID_LABELS = {
    'ACCESS-CM2':    'gn',
    'GFDL-ESM4':     'gr1',
    'MPI-ESM1-2-HR': 'gn',
    'NorESM2-MM':    'gn',
    'IPSL-CM6A-LR':  'gr',
}

SCENARIOS   = ['ssp245', 'ssp585']
ALL_VARS    = ['tas', 'tasmin', 'tasmax', 'pr']
HIST_YEARS  = list(range(2001, 2015))
HORIZONS    = {
    '2035': list(range(2026, 2046)),
    '2045': list(range(2036, 2056)),
}

# Big Island bounding box — NEX-GDDP uses 0–360 longitude
LAT_MIN, LAT_MAX       = 18.9, 20.3
LON_MIN_360, LON_MAX_360 = 360 - 156.1, 360 - 154.7

ENSEMBLE = 'r1i1p1f1'
S3_BUCKET = 'nex-gddp-cmip6'

fs = s3fs.S3FileSystem(anon=True)
print('Config OK')

In [ ]:
# ── Extraction helper ─────────────────────────────────────────────────────────
# For each model/scenario/variable/year: open NetCDF lazily from S3,
# slice to Big Island, compute 12-month climatology.
# Caches per-variable climatologies as pickles in OUT_DIR.

def s3_path(model, scenario, variable, year):
    grid  = GRID_LABELS.get(model, 'gn')
    fname = f'{variable}_day_{model}_{scenario}_{ENSEMBLE}_{grid}_{year}.nc'
    return f'{S3_BUCKET}/NEX-GDDP-CMIP6/{model}/{scenario}/{ENSEMBLE}/{variable}/{fname}'

def extract_clim(model, scenario, variable, years, label):
    """Return monthly climatology DataArray (12,) for Big Island.
    Averages spatially over the region then temporally over all years."""
    cache = os.path.join(OUT_DIR, f'{model}_{scenario}_{variable}_{label}.pkl')
    if os.path.exists(cache):
        return pd.read_pickle(cache)

    year_monthly = []   # list of (12,) arrays
    for year in years:
        path = s3_path(model, scenario, variable, year)
        try:
            with fs.open(path, 'rb') as fobj:
                ds = xr.open_dataset(fobj, engine='h5netcdf')
                lon_dim = 'lon' if 'lon' in ds.dims else 'longitude'
                lat_dim = 'lat' if 'lat' in ds.dims else 'latitude'
                roi = ds.sel({lat_dim: slice(LAT_MIN, LAT_MAX),
                              lon_dim: slice(LON_MIN_360, LON_MAX_360)})
                # Spatial mean over region, then monthly mean
                monthly = (roi[variable]
                           .mean(dim=[lat_dim, lon_dim])
                           .groupby('time.month')
                           .mean('time')
                           .load()
                           .values)   # shape (12,)
                year_monthly.append(monthly)
        except Exception as e:
            print(f'    SKIP {model}/{scenario}/{variable}/{year}: {e}')
            continue

    if not year_monthly:
        return None

    result = pd.Series(np.nanmean(year_monthly, axis=0),
                       index=range(1, 13), name=variable)
    pd.to_pickle(result, cache)
    return result

print('Helper defined.')

In [ ]:
# ── Run extraction + save per-model delta CSVs ────────────────────────────────
# For each model, scenario, horizon:
#   load historical + future climatology per variable
#   compute delta (additive for temp, multiplicative for precip)
#   save as CSV: {model}_{scenario}_{horizon}.csv
#      columns: month, delta_tas, delta_tasmin, delta_tasmax, ratio_pr
#
# The Big Island is ~6×6 GCM grid cells so spatial averaging is fast.
# Per-variable caches mean you can interrupt and resume safely.

KELVIN = 273.15

for model in MODELS:
    model_slug = model.replace('-', '_')
    print(f'\n=== {model} ===')

    # Load historical climatology
    hist = {}
    for var in tqdm(ALL_VARS, desc='  historical'):
        hist[var] = extract_clim(model, 'historical', var, HIST_YEARS, 'hist')

    if all(v is None for v in hist.values()):
        print(f'  No historical data — skipping {model}')
        continue

    for scenario in SCENARIOS:
        for horizon, years in HORIZONS.items():
            out_csv = os.path.join(OUT_DIR, f'{model_slug}_{scenario}_{horizon}.csv')
            if os.path.exists(out_csv):
                print(f'  cached: {os.path.basename(out_csv)}')
                continue

            futr = {}
            for var in tqdm(ALL_VARS, desc=f'  {scenario} {horizon}'):
                futr[var] = extract_clim(model, scenario, var, years, f'{scenario}_{horizon}')

            rows = []
            for month in range(1, 13):
                def g(d, var, m):
                    s = d.get(var)
                    return float(s[m]) if s is not None else np.nan

                # Temperature delta (Kelvin cancels — same as °C change)
                h_tas    = g(hist, 'tas', month)
                f_tas    = g(futr, 'tas', month)
                h_tasmin = g(hist, 'tasmin', month)
                f_tasmin = g(futr, 'tasmin', month)
                h_tasmax = g(hist, 'tasmax', month)
                f_tasmax = g(futr, 'tasmax', month)

                # If tas missing, approximate as (tasmin+tasmax)/2
                if np.isnan(h_tas) and not np.isnan(h_tasmin):
                    h_tas = (h_tasmin + h_tasmax) / 2
                    f_tas = (f_tasmin + f_tasmax) / 2

                # Precip ratio (kg m-2 s-1 units cancel in ratio)
                h_pr = g(hist, 'pr', month)
                f_pr = g(futr, 'pr', month)
                ratio_pr = float(np.clip(f_pr / (h_pr + 1e-12), 0.1, 5.0)) if not np.isnan(h_pr) else 1.0

                rows.append({
                    'month':        month,
                    'delta_tas':    f_tas    - h_tas,
                    'delta_tasmin': f_tasmin - h_tasmin,
                    'delta_tasmax': f_tasmax - h_tasmax,
                    'ratio_pr':     ratio_pr,
                })

            pd.DataFrame(rows).to_csv(out_csv, index=False)
            print(f'  saved: {os.path.basename(out_csv)}')

print('\nDone. CSVs saved to:', OUT_DIR)

In [ ]:
# ── List outputs ──────────────────────────────────────────────────────────────
csvs = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('.csv')])
print(f'{len(csvs)} delta CSVs in {OUT_DIR}:')
for f in csvs:
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) // 1024
    print(f'  {f}  ({size_kb} KB)')

print()
print('Next steps:')
print('  1. Download these CSVs from Google Drive')
print('  2. Copy to ../data/climate_projections/ on your laptop')
print('  3. Continue in 08_climate_projections.ipynb cell 5')